# 5. 为什么监管更偏爱 ES：次可加性（两笔贷款的例子）

**这一本要回答的问题：** 第 1 本已经看到一个理由："VaR 看不见门后面有多深"。这一本讲第二个、也是更"数学"的理由：**VaR 有时会说"分散投资让风险变大了"**，这违反常识；ES 永远不会。

先用一个只有两笔贷款的玩具例子把问题讲透，再回到我们自己的组合上看真实数据。

In [1]:
from learn_utils import *
from scipy.stats import binom, norm
from mre.risk.var import historical_var, historical_es

cfg, asset_rets, port = load_portfolio()
V = cfg.portfolio_value

## 第 1 步：直觉——"合在一起不应该更危险"

假设一家银行有两个部门 A 和 B。风险经理分别算出 A 的风险和 B 的风险，然后再把两个部门合起来算一次。常识告诉我们：

> **合起来的风险 ≤ A 的风险 + B 的风险**

因为两个部门不太可能**同时**碰上最坏的情况，合在一起会互相抵消一部分——这就是"分散化"。最坏也就是两者完全同步，那时正好相等。

一个风险指标如果满足这一点，就叫做**次可加**（subadditive）。听起来理所当然，但 **VaR 不一定满足**。

## 第 2 步：两笔贷款

- 有两笔**互相独立**的贷款，每笔借出 100 元。
- 每笔有 **4%** 的概率违约（亏 100 元），96% 的概率正常还款（亏 0 元）。
- 我们用 **95%** 的置信水平。

先用脑子算：

**单独一笔贷款的 95% VaR：** 最坏的 5% 情况里，有 4% 是"亏 100"，剩下 1% 是"亏 0"。数到 5% 的那个位置，落在"亏 0"上。所以 **VaR = 0**。VaR 说这笔贷款"没有风险"！

**两笔一起的 95% VaR：** "至少一笔违约"的概率 = 1 − 0.96 × 0.96 = 7.84%，已经超过 5% 了，数到 5% 的位置落在"亏 100"上。所以 **VaR = 100**。

> 单独算：0 + 0 = 0；合起来算：100。**VaR 说"分散投资之后风险从 0 变成了 100"。** 这就是违反次可加性。

下面用代码把所有可能的结果列出来，精确地算一遍。

In [2]:
def var_es_discrete(losses, probs, alpha):
    # 结果只有有限几种时的 VaR 和 ES（"按概率质量切"）。losses: 每种结果的损失；probs: 概率
    order = np.argsort(losses)
    L, P = np.asarray(losses, float)[order], np.asarray(probs, float)[order]
    cum = np.cumsum(P)
    var = L[np.argmax(cum >= alpha - 1e-12)]          # 从小到大累加概率，第一次 >= alpha 的那个损失
    tail = 1 - alpha                                  # 最坏的那 5% 概率质量
    beyond = L > var
    mass_beyond = P[beyond].sum()                     # 严格比 VaR 还惨的部分
    es = (np.sum(L[beyond] * P[beyond]) + (tail - mass_beyond) * var) / tail  # 不够 5% 的部分用 VaR 补齐
    return var, es

alpha = 0.95
pd_ = 0.04                                            # 违约概率

one = ([0, 100], [1 - pd_, pd_])
two = ([0, 100, 200], [(1 - pd_)**2, 2 * pd_ * (1 - pd_), pd_**2])

print("两笔贷款合起来，所有可能的结果：")
for l, p in zip(*two):
    print(f"  亏 {l:>3} 元，概率 {p:.2%}")

v1, e1 = var_es_discrete(*one, alpha)
v2, e2 = var_es_discrete(*two, alpha)
print(f"\n               单独一笔   两笔之和(单独算)   两笔合起来算")
print(f"95% VaR      {v1:8.1f}   {2 * v1:14.1f}   {v2:12.1f}   -> {'满足' if v2 <= 2 * v1 else '违反'}次可加性")
print(f"95% ES       {e1:8.1f}   {2 * e1:14.1f}   {e2:12.1f}   -> {'满足' if e2 <= 2 * e1 else '违反'}次可加性")

两笔贷款合起来，所有可能的结果：
  亏   0 元，概率 92.16%
  亏 100 元，概率 7.68%
  亏 200 元，概率 0.16%

               单独一笔   两笔之和(单独算)   两笔合起来算
95% VaR           0.0              0.0          100.0   -> 违反次可加性
95% ES           80.0            160.0          103.2   -> 满足次可加性


**ES 是怎么算出来的（手算一遍）：**

- 单独一笔：最坏的 5% 里，4% 是亏 100、1% 是亏 0，平均 = (4% × 100 + 1% × 0) ÷ 5% = **80**。
- 两笔合起来：最坏的 5% 里，0.16% 是亏 200，剩下 4.84% 是亏 100，平均 = (0.16% × 200 + 4.84% × 100) ÷ 5% = **103.2**。
- 103.2 ≤ 80 + 80 = 160，ES 认为分散化**降低**了风险，符合常识。

VaR 之所以出错，是因为单独一笔贷款的违约概率（4%）正好"躲"在 5% 的门槛下面，VaR 完全看不到它。合在一起之后违约概率超过了门槛，VaR 才突然"看见"。ES 一直在看门后面的平均，所以不会被这种门槛效应骗到。

### 用模拟验证一下（不信公式的话）

In [3]:
rng = np.random.default_rng(1)
n_sim = 1_000_000
default_a = rng.random(n_sim) < pd_
default_b = rng.random(n_sim) < pd_
loss_two = 100 * default_a + 100 * default_b
print(f"模拟 {n_sim:,} 次：至少一笔违约的比例 = {(loss_two > 0).mean():.2%}（理论 7.84%）")
print(f"最坏 5% 的模拟结果的平均损失 = {np.sort(loss_two)[-int(n_sim * 0.05):].mean():.1f}（理论 103.2）")

模拟 1,000,000 次：至少一笔违约的比例 = 7.89%（理论 7.84%）
最坏 5% 的模拟结果的平均损失 = 103.3（理论 103.2）


> **一个小细节：** 仓库里的 `historical_es` 是"把亏损 ≥ VaR 的所有日子取平均"。对真实的每日盈亏（几乎不会有两天亏得一模一样）这完全没问题；但在这种只有 0 / 100 / 200 几个取值的玩具例子里，"等于 VaR"的点有一大堆，直接套用会算错。所以这里用了上面"按概率质量正好切 5%"的写法。

## 第 3 步：🎛️ 你来调

把贷款数量、违约概率、置信水平都变成旋钮。每笔贷款还是 100 元、互相独立。看看什么时候 VaR 会违反次可加性：

In [4]:
def loans_lab(n_loans=2, default_prob=0.04, alpha=0.95):
    k = np.arange(n_loans + 1)                       # 违约的笔数：0, 1, ..., n
    losses, probs = 100 * k, binom.pmf(k, n_loans, default_prob)
    v1, e1 = var_es_discrete([0, 100], [1 - default_prob, default_prob], alpha)
    vn, en = var_es_discrete(losses, probs, alpha)
    print(f"{n_loans} 笔贷款，每笔违约概率 {default_prob:.1%}，置信水平 {alpha:.0%}")
    print(f"  VaR: 单独算再加总 = {n_loans * v1:7.1f}，合起来算 = {vn:7.1f}  -> {'满足' if vn <= n_loans * v1 + 1e-9 else '违反！'}")
    print(f"  ES : 单独算再加总 = {n_loans * e1:7.1f}，合起来算 = {en:7.1f}  -> {'满足' if en <= n_loans * e1 + 1e-9 else '违反！'}")

loans_lab(2, 0.04, 0.95)

2 笔贷款，每笔违约概率 4.0%，置信水平 95%
  VaR: 单独算再加总 =     0.0，合起来算 =   100.0  -> 违反！
  ES : 单独算再加总 =   160.0，合起来算 =   103.2  -> 满足


In [5]:
play(loans_lab, examples=[{"n_loans": 2, "default_prob": 0.06, "alpha": 0.95}, {"n_loans": 2, "default_prob": 0.04, "alpha": 0.99}, {"n_loans": 10, "default_prob": 0.04, "alpha": 0.95}],
     n_loans=(1, 20, 1), default_prob=(0.005, 0.10, 0.005), alpha=[0.90, 0.95, 0.975, 0.99])

（在 Jupyter 里这里是滑块。静态版本里换成下面几组参数各跑一遍。）

▶ n_loans = 2，default_prob = 0.06，alpha = 0.95
2 笔贷款，每笔违约概率 6.0%，置信水平 95%
  VaR: 单独算再加总 =   200.0，合起来算 =   100.0  -> 满足
  ES : 单独算再加总 =   200.0，合起来算 =   107.2  -> 满足

▶ n_loans = 2，default_prob = 0.04，alpha = 0.99
2 笔贷款，每笔违约概率 4.0%，置信水平 99%
  VaR: 单独算再加总 =   200.0，合起来算 =   100.0  -> 满足
  ES : 单独算再加总 =   200.0，合起来算 =   116.0  -> 满足

▶ n_loans = 10，default_prob = 0.04，alpha = 0.95
10 笔贷款，每笔违约概率 4.0%，置信水平 95%
  VaR: 单独算再加总 =     0.0，合起来算 =   200.0  -> 违反！
  ES : 单独算再加总 =   800.0，合起来算 =   213.4  -> 满足


试着回答：
1. 违约概率调到 6%（超过了 5% 的门槛），两笔贷款时 VaR 还违反吗？
2. 违约概率 4%、置信水平改成 99%，会怎样？
3. 不管你怎么调，ES 那一行有没有出现过"违反"？

## 第 4 步：回到我们自己的组合

真实的组合里，VaR 违反次可加性的情况其实不常见（通常发生在像贷款、期权这种"要么没事、要么大亏"的产品上）。但**不常见不等于不会**，而 ES 是**数学上保证**永远满足。

下面把我们的组合拆成 4 个"部门"（SPY、QQQ、TLT、GLD 各自那部分仓位），分别算 99% VaR 和 ES，再和整个组合比：

In [6]:
def diversification(alpha=0.99, window="全部 20 年"):
    seg = {"全部 20 年": slice(None), "2008 年": slice("2008-01-01", "2008-12-31"),
           "2020 年": slice("2020-01-01", "2020-12-31"), "2022 年": slice("2022-01-01", "2022-12-31")}[window]
    sleeves = asset_rets.loc[seg] * pd.Series(cfg.weights) * V     # 每个"部门"每天的盈亏
    total = sleeves.sum(axis=1)                                      # 整个组合每天的盈亏
    rows = [{"部门": c, "VaR": historical_var(sleeves[c], alpha), "ES": historical_es(sleeves[c], alpha)}
            for c in sleeves]
    t = pd.DataFrame(rows).set_index("部门")
    t.loc["单独算再加总"] = t.sum()
    t.loc["整个组合"] = [historical_var(total, alpha), historical_es(total, alpha)]
    t.loc["分散化省下的比例"] = 1 - t.loc["整个组合"] / t.loc["单独算再加总"]
    out = t.copy().astype(object)
    for idx in t.index:
        out.loc[idx] = [f"{x:.1%}" if idx == "分散化省下的比例" else money(x) for x in t.loc[idx]]
    print(f"{window}，{len(total)} 天，{alpha:.0%} 置信水平")
    return out

diversification(0.99, "全部 20 年")

全部 20 年，5030 天，99% 置信水平


,VaR,ES
部门,,
SPY,"$14,284","$20,812"
QQQ,"$11,749","$15,513"
TLT,"$4,653","$5,917"
GLD,"$3,073","$4,284"
单独算再加总,"$33,759","$46,526"
整个组合,"$24,197","$34,116"
分散化省下的比例,28.3%,26.7%


In [7]:
play(diversification, examples=[{"alpha": 0.99, "window": "2022 年"}, {"alpha": 0.99, "window": "2008 年"}],
     alpha=[0.95, 0.975, 0.99], window=["全部 20 年", "2008 年", "2020 年", "2022 年"])

（在 Jupyter 里这里是滑块。静态版本里换成下面几组参数各跑一遍。）

▶ alpha = 0.99，window = 2022 年
2022 年，251 天，99% 置信水平


,VaR,ES
部门,,
SPY,"$14,985","$16,235"
QQQ,"$14,338","$15,429"
TLT,"$5,373","$6,197"
GLD,"$1,996","$2,578"
单独算再加总,"$36,691","$40,439"
整个组合,"$32,784","$35,973"
分散化省下的比例,10.7%,11.0%



▶ alpha = 0.99，window = 2008 年
2008 年，253 天，99% 置信水平


,VaR,ES
部门,,
SPY,"$30,486","$35,385"
QQQ,"$20,720","$24,412"
TLT,"$4,540","$5,337"
GLD,"$5,340","$6,305"
单独算再加总,"$61,086","$71,439"
整个组合,"$41,720","$55,023"
分散化省下的比例,31.7%,23.0%


在我们的组合上 VaR 和 ES 都显示分散化省了不少风险（TLT 和 GLD 在股市大跌时常常不跟着跌）。🎛️ 切到 **2022 年** 看看：那一年股票和长期国债一起跌，分散化的效果还那么好吗？

## 第 5 步：监管的选择——用 97.5% ES 代替 99% VaR

巴塞尔委员会的交易账簿新规（FRTB）把市场风险资本的核心指标从 **99% VaR** 换成了 **97.5% ES**。为什么是 97.5% 而不是 99%？因为如果收益是正态分布，97.5% ES 和 99% VaR 几乎一样大：

In [8]:
print(f"正态分布下（以标准差为单位）：99% VaR = {norm.ppf(0.99):.3f}，97.5% ES = {norm.pdf(norm.ppf(0.975)) / 0.025:.3f}")
pnl = port * V
print(f"\n我们组合的 20 年真实数据（历史模拟）：")
print(f"  99% VaR  = {money(historical_var(pnl, 0.99))}")
print(f"  97.5% ES = {money(historical_es(pnl, 0.975))}")

正态分布下（以标准差为单位）：99% VaR = 2.326，97.5% ES = 2.338

我们组合的 20 年真实数据（历史模拟）：
  99% VaR  = $24,197
  97.5% ES = $25,828


所以监管的思路是：**在"正常"（正态）的世界里，换指标不会改变资本要求；但如果尾巴比正态更肥，ES 会自动要求更多资本。** 我们的组合尾巴比正态肥（Phase 1 报告：超额峰度 11.11），所以 97.5% ES 比 99% VaR 更大。

## 最后：公式和"一致性风险度量"

用 $\rho(\cdot)$ 表示"一个风险指标"（比如 VaR 或 ES），$X$、$Y$ 表示两个部门的损失。学术上（Artzner 等人，1999 年）认为一个"好"的风险指标应该满足四条，满足四条的叫**一致性风险度量**（coherent risk measure）：

| 性质 | 大白话 | VaR | ES |
|---|---|---|---|
| 单调性 | 如果 X 在任何情况下都比 Y 亏得多，X 的风险就更大 | ✓ | ✓ |
| 正齐次性 | 仓位翻倍，风险也翻倍：$\rho(2X) = 2\rho(X)$ | ✓ | ✓ |
| 平移不变性 | 多放 100 元现金，风险就少 100 元 | ✓ | ✓ |
| **次可加性** | **合起来不比分开更危险**：$\rho(X+Y) \le \rho(X) + \rho(Y)$ | **✗ 不一定** | **✓ 永远** |

VaR 只差最后一条，而这一条正好就是"分散化"这个金融里最基本的想法。这就是 ES 被称为"一致的"而 VaR 不是的原因。

ES 的另一种写法（对理解"为什么它永远次可加"有帮助）：ES 就是把 $\alpha$ 到 1 之间**所有**置信水平的 VaR 取平均，

$$\text{ES}_\alpha = \frac{1}{1-\alpha}\int_\alpha^1 \text{VaR}_u \, du$$

积分号 $\int$ 在这里就是"连续地求平均"。只看一个点（VaR）会被门槛骗到，看一整段的平均就不会。

**对应代码：** `src/mre/risk/var.py`；组合拆成"部门"算风险贡献的正规做法在 `src/mre/attribution/euler.py`（Phase 4 的欧拉分解），那是另一个话题了。

## 小练习

1. 用自己的话，向一个不懂金融的朋友解释"次可加性"。
2. 为什么"要么没事、要么大亏"的产品（贷款、卖出的深度虚值期权）最容易让 VaR 出问题？
3. ES 有没有缺点？

<details><summary>点这里看参考答案</summary>

1. "把两个篮子的鸡蛋合到一起，打碎的风险不应该比两个篮子分开时加起来还大。"
2. 它们的大亏概率往往很小，可能正好躲在 VaR 的门槛下面（比如 4% < 5%），单独看 VaR 是 0，合起来概率超过门槛就突然跳出来。
3. 有。ES 依赖极端尾部的少数观测，估计误差比 VaR 大；而且 ES 不"可引出"（elicitable），回测比 VaR 难——这也是为什么仓库的回测（第 4 本）主要检验 VaR，ES 只用 `es_ratio` 做了描述性检查。
</details>